## Running VGAE / Graph-PCA & baseline models

- Dim-reduction on features ($x \in \mathbb{R}^{N \times P}$: observation $z \in \mathbb{R}^{N \times D}$: representation, $u \in \mathbb{R}^{N \times 1}$: interpretability as "trajectory / gradient")
- Reconstruction w/ $\sigma(z \cdot z^T)$ for graph reconstruction, regularize $u$ w/ CyCIF graph prior
- Benchmark against K-means clustering & PCA, regular VAE, etc.

In [ ]:
import os
import gc
import sys
import pickle
import gzip

import numpy as np
import cupy as cp
import pandas as pd
import scanpy as sc

import torch
import tifffile
import torch.nn as nn

import networkx as nx
import tifffile
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy import sparse
from torch_geometric import utils as pyg_utils

torch.manual_seed(42)

In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, utils, plot, gen_graph, configs, dataset, zonation
import baseline, sb_vgae, model_train, model_eval

from torch_geometric.loader import DataLoader

### VGAE (Xenium-DESI integration)

- Integrate DESI prior to guide latent prob. clustering of Xenium

In [ ]:
import json
import squidpy as sq

from sklearn.decomposition import FastICA
from util import IO, utils, gen_graph
from models import baseline, dataset
from torch_geometric import utils as pyg_utils

from importlib import reload
reload(IO)
reload(utils)
reload(baseline)
reload(gen_graph)
reload(dataset)

torch.manual_seed(42)

In [ ]:
# Load paired Xenium & DESI

xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'
sample_id = 'NIH_F5'
# sample_ids = sorted([f for f in os.listdir(xenium_path) if os.path.isdir(os.path.join(xenium_path, f))])

# Load xenium
adata = IO.load_xenium(os.path.join(xenium_path, sample_id))
with open(os.path.join('../data/xenium/', sample_id, 'experiment.xenium'), 'r') as ifile:
    scalefactor = json.load(ifile)['pixel_size']
xenium_coords = adata.obs[['y_centroid', 'x_centroid']].copy().to_numpy() 


In [ ]:
# Load DESI pixels mapped to Xenium
ratio = 1/10  # Xenium -> DESI downscale ratio
adata_desi = IO.load_paired_desi(os.path.join(desi_path, sample_id+'.ome.tif'),
                                 adata_other=adata,
                                 ratio=ratio / scalefactor,
                                 dilate=True)
desi_coords = adata_desi.obs[['y_centroid', 'x_centroid']].copy().to_numpy()
                                 

In [ ]:
# Compute aux. DESI expressions `u`
# Experiment: Encoding w/ PCA or ICA

def get_PCs(adata, coords, 
            n_pcs, k=8, 
            graph_regularize=False):
    """
    Dimension reduction w/ (graph-regularized) PCA
    """
    if graph_regularize:
        G = gen_graph.construct_graph(coords, k=k, weighted=False)  
        edge_index = pyg_utils.from_networkx(G).edge_index
        model = baseline.GPCALayer(c_in=adata.X.shape[-1],
                                   c_out=n_pcs,
                                   center=True,
                                   init_weight=True,
                                   ortho_weight=True)
        U_gpca = model(torch.tensor(adata.X).float(), edge_index)
        adata.obsm['X_pca'] = U_gpca.detach().cpu().numpy()
    else:
        sc.pp.pca(adata, n_pcs)
    return None

# n_aux = 30
# get_PCs(adata_desi, desi_coords, 
#         n_pcs=n_aux,
#         graph_regularize=True)

# sc.pp.neighbors(adata_desi, n_neighbors=30, use_rep='X_pca')
# sc.tl.leiden(adata_desi, resolution=0.1)
# print(adata_desi.obs.leiden.value_counts())
# sc.pl.spatial(adata_desi, color='leiden', size=5)

# Append aux. DESI expression (PC) to Xenium
# adata.obsm['X_aux'] = adata_desi.obsm['X_pca']  

n_aux = 16
ica_model = FastICA(n_components=n_aux, random_state=0, whiten='unit-variance')
s = ica_model.fit_transform(adata_desi.X)
adata_desi.obsm['X_ica'] = s

adata.obsm['X_u'] = adata_desi.X
adata.obsm['X_aux'] = s

#### Model sketch iVAE: 
- $z_i \mid u_i \sim \mathcal{LN}(f_{\lambda}(u_i), \sigma^2I)$
- $x_i \mid z_i, \mathbf{A} \sim \mathcal{NegBinom}(l \cdot f_{\theta}(z_i, \mathbf{A}, \theta_g)$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import pyro
import pyro.poutine as poutine
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, TraceEnum_ELBO
from pyro.optim import Adam

from torch_geometric.data import DataLoader

from models import logit_vgae, model_train

In [ ]:
xenium_loader = dataset.XeniumGraphDataset(k=30, n_subgraphs=8) 
xenium_data = xenium_loader.load_graphs([adata])
xenium_train_dl = DataLoader(xenium_data, shuffle=True)

Observation: Linear ICA demix DESI well

In [ ]:
from sklearn.decomposition import FastICA

ica_model = FastICA(n_components=16, random_state=0, whiten='unit-variance')
s_desi = ica_model.fit_transform(adata_desi.X)
for i in range(s_desi.shape[1]):
    adata_desi.obs['IC'+str(i)] = s_desi[:, i]

ic_labels = ['IC'+str(i) for i in range(s_desi.shape[1])]
sq.pl.spatial_scatter(adata_desi, color=ic_labels, cmap='coolwarm', size=20, img=False)

del ica_model, s_desi, ic_labels

In [ ]:
from importlib import reload

reload(IO)
reload(utils)
reload(plot)
reload(dataset)
reload(configs)
reload(logit_vgae)
reload(model_train)

In [ ]:
lr = 1e-3
n_epochs = 1000
device = torch.device('cuda')
n_aux = adata.obsm['X_aux'].shape[1]

train_configs = configs.set_train_configs(n_epochs=n_epochs, 
                                          lr=lr,
                                          annealing=False,
                                          device=device)

model_configs = configs.set_model_configs(device=device,
                                          beta=0.5,
                                          c_in=adata.shape[1],
                                          c_u=adata_desi.shape[1],
                                          c_aux=n_aux,
                                          c_hidden=16,
                                          c_latent=7,
                                          k_hop=3,
                                          dropout=0.5) 

In [ ]:
pyro.clear_param_store()
model = logit_vgae.LogitVGAE(model_configs, device=train_configs.device)
model, losses = model_train.train_logit_vgae(model, xenium_train_dl, train_configs)

In [ ]:
plt.figure(figsize=(5, 2))
plt.plot(np.arange(train_configs.n_epochs), losses)
plt.xlabel('Epochs')
plt.ylabel('Train ELBO')
plt.show()

In [ ]:
# Inference on full data (CPU)

G_xenium = gen_graph.construct_graph(xenium_coords, k=30, r=np.inf)
edge_index = pyg_utils.from_networkx(G_xenium).edge_index

model.device = 'cpu'
model = model.to('cpu')

qz, qz_mu, qz_logstd = model.get_z(torch.tensor(adata.X.A).to('cpu'), 
                                   torch.tensor(adata.obsm['X_u']).to('cpu'), 
                                   edge_index.to('cpu'))

pz = model.get_cond_prior(torch.tensor(adata.obsm['X_aux']).to('cpu'),
                          edge_index.to('cpu'))
px = model.get_x(torch.tensor(adata.X.A).to('cpu'), 
                 edge_index.to('cpu'), 
                 qz_mu, qz_logstd)

qz_mu, qz_logstd = qz_mu.detach().cpu().numpy(), qz_logstd.detach().cpu().numpy()
qz = qz.detach().cpu().numpy()

pz = pz.detach().cpu().numpy()
px = px.detach().cpu().numpy()

# del G_xenium
gc.collect()
torch.cuda.empty_cache()

In [ ]:
rand_indices = np.random.choice(np.arange(adata.shape[1]), size=100, replace=False)

plt.figure(figsize=(6, 4))
plt.scatter(adata.X.A[:, rand_indices].flatten(), px[:, rand_indices].flatten(), s=1)
plt.xlabel('X')
plt.ylabel("X_reconst")
plt.show()

del rand_indices

In [ ]:
sns.heatmap(np.corrcoef(pz.T), cmap='coolwarm')

In [ ]:
sns.clustermap(np.corrcoef(qz.T), cmap='coolwarm')

In [ ]:
# Check posterior collapse?
plot.disp_spatial_latents(adata, pz, ncols=4)

In [ ]:
plot.disp_spatial_latents(adata, qz, ncols=4)

**DEBUG**: <br>

$p(z \mid u)$ stable & disentangled, but $q(z \mid x, u)$ persists w/ identical latent factors
- Not fitting well (KL term?)

#### Trajectory Inference

- Try both Xenium & mapped DESI

In [ ]:
# Load saved latent representations
adata_qz = sc.read_h5ad('../data/xenium/adata_NIH_F5_qz.h5ad')
qz = adata_qz.X

In [ ]:
# adata_qz = sc.AnnData(qz)
# sc.pp.pca(adata_qz)
# sc.pp.neighbors(adata_qz, n_neighbors=30)
# sc.tl.umap(adata_qz, n_components=3)

adata_norm = adata.copy()
sc.pp.normalize_total(adata_norm)
sc.pp.log1p(adata_norm)
sq.gr.spatial_neighbors(adata_norm, coord_type='generic', spatial_key='spatial')  # Spatial corr.
sq.gr.spatial_autocorr(adata_norm, mode='moran')  
# sc.pp.pca(adata_norm)


# # Assign learnt latent embeddings to Xenium / DESI dataset as metadata
adata_norm.obsm['X_pca'] = adata_qz.obsm['X_pca']
adata_norm.obsm['X_umap'] = adata_qz.obsm['X_umap']
adata_norm.obsm['X_z'] = qz

adata_desi.obsm['X_pca'] = adata_qz.obsm['X_pca']
adata_desi.obsm['X_umap'] = adata_qz.obsm['X_umap']
adata_desi.obsm['X_z'] = qz


In [ ]:
# PC on raw Xenium expressions

# sc.pp.pca(adata_norm)
# sc.pl.pca(adata_norm,
#           color=['ADH1C', 'CYP3A4', 'APOA5', 'AQP9',  # PC markers
#                  'CYP2A7', 'CYP2B6', 'HPX', 'HMGCS2',  # PP markers
#                  'DPT', 'PTGDS', 'MYH11', 'CD34' # Portal-related
#                 ],
#           size=10, 
#           cmap='RdBu_r')


In [ ]:
# Xenium markers
sc.pl.pca(adata_norm,
          color=['ADH1C', 'CYP3A4', 'APOA5', 'AQP9',  # PC markers
                 'CYP2A7', 'CYP2B6', 'HPX', 'HMGCS2',  # PP markers
                 'DPT', 'PTGDS', 'MYH11', 'CD34' # Portal-related
                ],
          size=20, 
          cmap='RdBu_r')


In [ ]:
sc.pl.umap(adata_norm,
          color=['ADH1C', 'CYP3A4', 'APOA5', 'AQP9',  # PC markers
                 'CYP2A7', 'CYP2B6', 'HPX', 'HMGCS2',  # PP markers
                 'DPT', 'PTGDS', 'MYH11', 'CD34' # Portal-related
                ],
          size=20, 
          cmap='RdBu_r')

In [ ]:
sc.pl.umap(adata_norm,
          color=['ADH1C', 'CYP3A4', 'APOA5', 'AQP9',  # PC markers
                 'CYP2A7', 'CYP2B6', 'HPX', 'HMGCS2',  # PP markers
                 'DPT', 'PTGDS', 'MYH11', 'CD34' # Portal-related
                ],
          size=20, 
          projection='3d', alpha=0.5,
          cmap='RdBu_r')

In [ ]:
# DESI markers
sc.pl.pca(adata_desi,
          color=['FA 20:4;O (C296)',
                 'FA 18:1; O (C286)',
                 'FA 22:5;O (C451)',
                 '808.6024 m/z ± 30 ppm (C40)',
                 '865.50838 m/z ± 50 ppm (C385)',
                 '673.49987 m/z ± 30 ppm (C10)',
                 '631.4937 m/z ± 30 ppm (C7)',
                 '346.05909 m/z ± 40 ppm (C126)',
                 '754.5545 m/z ± 30 ppm (C18)',
                 '258.11631 m/z ± 30 ppm (C1)',
                 '734.58637 m/z ± 30 ppm (C16)',
                 '703.58429 m/z ± 30 ppm (C13)',
                ],
          size=20, 
          cmap='RdBu_r')

In [ ]:
sc.pl.umap(adata_desi,
          color=['FA 20:4;O (C296)',
                 'FA 18:1; O (C286)',
                 'FA 22:5;O (C451)',
                 '808.6024 m/z ± 30 ppm (C40)',
                 '865.50838 m/z ± 50 ppm (C385)',
                 '673.49987 m/z ± 30 ppm (C10)',
                 '631.4937 m/z ± 30 ppm (C7)',
                 '346.05909 m/z ± 40 ppm (C126)',
                 '754.5545 m/z ± 30 ppm (C18)',
                 '258.11631 m/z ± 30 ppm (C1)',
                 '734.58637 m/z ± 30 ppm (C16)',
                 '703.58429 m/z ± 30 ppm (C13)',
                ],
          size=20, 
          cmap='RdBu_r')

In [ ]:
sc.pl.umap(adata_desi,
          color=['FA 20:4;O (C296)',
                 'FA 18:1; O (C286)',
                 'FA 22:5;O (C451)',
                 '808.6024 m/z ± 30 ppm (C40)',
                 '865.50838 m/z ± 50 ppm (C385)',
                 '673.49987 m/z ± 30 ppm (C10)',
                 '631.4937 m/z ± 30 ppm (C7)',
                 '346.05909 m/z ± 40 ppm (C126)',
                 '754.5545 m/z ± 30 ppm (C18)',
                 '258.11631 m/z ± 30 ppm (C1)',
                 '734.58637 m/z ± 30 ppm (C16)',
                 '703.58429 m/z ± 30 ppm (C13)',
                ],
          size=20,
          projection='3d', alpha=0.5,
          cmap='RdBu_r')

In [ ]:
adata_qz.write_h5ad('../data/xenium/adata_NIH_F5_qz.h5ad')

**Wasserstein Distance**<br>  (Archived! directly compute principal curve)


---

In [ ]:
import networkx
from scipy.stats import wasserstein_distance

In [ ]:
# Compute cluster-level "trajectory" via DFS on NNs?
def dfs(mat, start, order='descending', visited=None):
    """
    Compute cluster-level "trajectory" via DFS 
    on nearest neighbor clusters
    """
    assert order == 'ascending' or order == 'descending'
    if visited is None:
        visited = []
    visited.append(start)
    
    # Visit all the neighbors
    neighbors = np.argsort(mat[start]) if order == 'ascending' else np.argsort(mat[start])[::-1]
    neighbors = np.delete(neighbors, np.argwhere(neighbors == start))  # avoid self
    for neighbor in neighbors:
        if neighbor not in visited:
            dfs(mat, neighbor, order, visited)
    
    return visited


def get_wass_dist(m, n):
    x = m / m.sum()
    y = n / n.sum()
    return wasserstein_distance(x, y)


def get_latent_dists(latent):
    """Relative `distance` btw clusters"""
    ndim = latent.shape[1]
    mat = np.zeros((ndim, ndim), dtype=np.float32)
    for i in range(ndim-1):
        for j in range(i+1, ndim):
            mat[i, j] = get_wass_dist(latent[:, i], latent[:, j])
    return mat+mat.T  # Complete Upp-tril to full symmetric matrix.


def sort_latent(adata, root_marker, term_marker=None):
    """
    Sort soft cluster assignments (z) by 
    Wasserstein distance to the root marker 
    """
    assert root_marker in adata.var_names, \
        "Root marker {} doesn't exist".format(root_marker)
    assert 'X_z' in adata.obsm, \
        "Please run the model first the obtain the latent representation"

    z = adata.obsm['X_z']
    dists = []
    root_expr = adata[:, root_marker].X
    term_expr = adata[:, term_marker].X if term_marker in adata.var_names else None
    if not isinstance(root_expr, np.ndarray):
        root_expr = root_expr.A
        if term_expr is not None:
            term_expr = term_expr.A
            
    for i, zi in enumerate(z.T):
        dist_to_root = get_wass_dist(root_expr.squeeze(), zi)
        if term_expr is None:
            dists.append(dist_to_root)
        else:
            dist_to_term = get_wass_dist(term_expr.squeeze(), zi)
            weighted_dist = dist_to_root / (dist_to_root + dist_to_term)
            dists.append(weighted_dist)

    return z[:, np.argsort(dists)]

In [ ]:
# Experiment: 
# (1). Sort by Wasserstein btw latent & marker gene(s)
# qz_sorted = sort_latent(adata_norm,
#                         root_marker='MYH11',
#                         term_marker='APOA5')


# (2). Sort by DFS on latent correlation matrix w/ Start
qz_sorted = qz[:, dfs(np.corrcoef(qz.T), start=0)]

z_labels = []
for i in range(qz.shape[1]):
    label = 'z'+str(i)
    adata_qz.obs[label] = qz_sorted[:, i]
    z_labels.append(label)
del label

sns.heatmap(np.corrcoef(qz_sorted.T), cmap='coolwarm')
plot.disp_spatial_latents(adata, qz_sorted, ncols=4)

In [ ]:
sc.pl.umap(adata_qz, color=z_labels, cmap='RdBu_r')

---

**Pseudotime sketch:**
- Fit an Elastic principal curve on `Z`
- Compute linear path btw principal "nodes"
- Compute k-NN distance from each cell to principal nodes --> final pseudotime assignment  

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import scFates as scf
import igraph

In [ ]:
n_pcurve_nodes = 20
scf.tl.curve(adata_qz, 
             Nodes=n_pcurve_nodes, 
             ndims_rep=adata_qz.shape[1])

In [ ]:
scf.pl.graph(adata_qz, basis='umap', nodes=list(np.arange(n_pcurve_nodes)))

`adata.uns['graph']['F']`: K x dim(Nodes) low-dim representation of the principal points <br>
`adata.uns['graph']['B']`: dim(Nodes) x dim(Nodes) adjacency matrix of the principal points <br>
`adata.obsm['X_R']`: soft assignment of cells to principal points

In [ ]:
def get_pcurve_path(adata):
    """Compute linear trajectory btw principal nodes in latent space"""
    assert 'graph' in adata.uns.keys(), "Please run Principal Curve first"

    al = np.array(
        igraph.Graph.Adjacency(
            (adata.uns['graph']['B'] > 0).tolist(), 
            mode='undirected'
        ).get_edgelist()
    )

    root_node, term_node = adata.uns['graph']['tips']
    curr_node = root_node
    ypos, xpos = np.asarray(np.where(al == root_node)).T[0]  # coords of current node
    
    path = [curr_node]
    while term_node not in path:
        xpos = 1 - xpos  # adjacenct principle node
        curr_node = al[ypos, xpos]
        path.append(curr_node)
        if curr_node == term_node:
            break
        
        # Update `ypos`, `xpos` of `curr_node`:
        # besides root & term node, every other node appears twice 
        coords = np.asarray(np.where(al == curr_node)).T  # dim: [2, 2]
        if np.array_equal(coords[0], [ypos, xpos]):
            ypos, xpos = coords[1]
        else:
            ypos, xpos = coords[0]
        
    return path


In [ ]:
from scanpy.tools._dpt import DPT
from scipy.spatial.distance import cdist

def get_diffusion_dist(adata, root_expr, n_neighbors=50):
    # Append "dummy" principal node
    adata_root = sc.AnnData(np.expand_dims(root_expr, 0))
    adata_cat = sc.concat((adata, adata_root), axis=0)

    sc.pp.neighbors(adata_cat, n_neighbors=n_neighbors)
    
    dpt = DPT(adata_cat)
    dpt.compute_transitions()
    dpt.compute_eigen(n_comps=adata_cat.shape[1])
    dpt.iroot = adata_cat.shape[0] - 1
    dpt._set_pseudotime()
    
    return dpt.pseudotime[:-1]  # Drop "dummy" principal node

def dist_to_pcurve(adata, n_neighbors=50):
    """
    Compute diffusion distance (DPT: D(x, y)) btw each cell (x) 
    and latent representation vector of each principal node (y)
    """ 
    assert 'graph' in adata.uns.keys(), "Please run Principal Curve first"
    
    pcurve_reprs = adata.uns['graph']['F'].T  # dim:[n_nodes, n_latent (K)]
    n_pts = adata.shape[0]
    n_nodes, n_latent = pcurve_reprs.shape
    dists = np.zeros((n_pts, n_nodes), dtype=np.float32)

    # Compute trajectory ordering of principal nodes
    pcurve_indices = get_pcurve_path(adata)
    pcurve_reprs = pcurve_reprs[pcurve_indices, :]

    # TODO: [DEBUG] directly compute Euclidean distance vs. Diffusion distance
    for i, pcurve in enumerate(pcurve_reprs):
        dists[:, i] = cdist(adata.X, np.expand_dims(pcurve, 0)).squeeze()
    #   dists[:, i] = get_diffusion_dist(adata, pcurve_repr[i], n_neighbors)
    
    return dists

In [ ]:
t = dist_to_pcurve(adata_qz)

# t_labels = ['t'+str(i) for i in range(t.shape[1])]
# for i, label in enumerate(t_labels):
#     adata_norm.obs[label] = t[:, i]  # Assign DPT to each node on full data
# del i, label

In [ ]:
sns.heatmap(np.corrcoef(t.T), cmap='coolwarm')

In [ ]:

adata_norm.obs['pseudotime'] = t.argmin(1)  # Discrete assignment
sc.pl.umap(adata_norm, color='pseudotime', cmap='RdBu')
sq.pl.spatial_scatter(adata_norm, color='pseudotime', cmap='tab20', size=20, img=False)
# adata_norm.obs.drop('pseudotime', axis=1, inplace=True)

**TI w/ scFates:**

In [ ]:
import scFates as scf

In [ ]:
# adata_qz = sc.read_h5ad('../data/xenium/adata_NIH_F5_qz.h5ad')

# adata_norm.obsm['X_z'] = adata_qz.obs.values  # Sorted qz
# adata_norm.obsm['X_pca'] = adata_qz.obsm['X_pca']

# adata_desi.obsm['X_z'] = adata_qz.obs.values
# adata_desi.obsm['X_pca'] = adata_qz.obsm['X_pca']

In [ ]:
adata_norm.obsm['X_z'] = qz_sorted
scf.tl.curve(adata_norm, Nodes=30, use_rep='X_z', ndims_rep=adata_norm.obsm['X_z'].shape[-1])

In [ ]:
# TI w/ root assignment
scf.tl.root(adata_norm, "MYH11")

In [ ]:
scf.tl.pseudotime(adata_norm, n_jobs=16, n_map=100, seed=0)

In [ ]:
sc.pl.pca(adata_norm, color="t", cmap="coolwarm", title="Pseudotime\n (principal curve)")

In [ ]:
# Spatial distribution of trajectories
sq.pl.spatial_scatter(adata_norm, color='t', size=20, img=False, cmap='coolwarm', title='Spatial Trajectory\n (principal curve)')

DEGs along trajectory:

In [ ]:
# Xenium
scf.tl.test_association(adata_norm, n_jobs=16)
scf.tl.test_association(adata_norm, reapply_filters=True,A_cut=.5)
scf.pl.test_association(adata_norm)
ti_sig_features = adata_norm.var[adata_norm.var.signi].index

In [ ]:
scf.tl.fit(adata_norm, n_jobs=16)

Held-out validation:

In [ ]:
# adata_val = load_xenium(os.path.join(xenium_path, 'NIH_F4'))
# xenium_coords = adata_val.obs[['y_centroid', 'x_centroid']].copy().to_numpy()

# TODO: need to load paired `u`

G_xenium = gen_graph.construct_graph(xenium_coords, k=30, r=np.inf)
edge_index = pyg_utils.from_networkx(G_xenium).edge_index
qz, qz_Sigma = model.get_z(adata_val.X.A, edge_index)

del G_xenium, edge_index
gc.collect()

In [ ]:
plot.disp_spatial_latents(adata_val, qz, ncols=4)

Baselines: EM clustering:

In [ ]:
from sklearn.mixture import GaussianMixture

gm = GaussianMixture(n_components=10, random_state=0).fit(adata_norm.X.A)
gm_cov = gm.covariances_
gm_soft_assign = gm.predict_proba(adata_norm.X.A)

plot.disp_spatial_latents(adata, gm_soft_assign, ncols=2)

---